In [141]:

# ANALISIS DE MÉTRICAS DE DIVERSIDAD
# Este script aplica un conjunto de enfoques complementarios para caracterizar
# la diversidad alfa en tu repertorio: primero calcula los números de Hill 
# mediante alphaDiversity de alakazam ; luego incorpora métricas adicionales 
# con vegan como índices de diversidad y equidad para describir la distribución
# de abundancias; y finalmente utiliza ineq para estimar desigualdad clonal a través
# del índice de Gini y otras medidas de concentración. Al combinar estas diez métricas, 
# obtienes una visión integrada de la cantidad, equilibrio y desigualdad en la arquitectura 
# clonal de tu repertorio.

In [142]:
# Paquetes y librerías
library(readr)
library(dplyr)
library(ggplot2)
library(viridisLite)
library(here)  # para rutas relativas
# install.packages("alakazam")
library(alakazam)
library(ineq)
library(vegan)
library(tidyr)
library(tibble)

In [143]:

# Cargar tu archivo .tsv
archivo_clones <- "../data/output/repertorio_E_insilico_1000_seqs_clone-pass.tsv"
clones <- read_tsv(archivo_clones)
# Agregar identificador de muestra (porque es simulado)
clones <- clones %>%
  mutate(sample_id = "repertorio_simulado")


# Contar secuencias por clon y muestra
clone_counts <- clones %>%
  group_by(sample_id, clone_id) %>%
  summarise(count = n(), .groups = "drop")

Rows: 1000 Columns: 50
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (20): sequence_id, sequence, v_call, d_call, j_call, sequence_alignment,...
dbl (25): junction_length, np1_length, np2_length, v_sequence_start, v_seque...
lgl  (5): rev_comp, productive, stop_codon, vj_in_frame, c_call

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [144]:
hill <- alphaDiversity(
  data = clone_counts,
  clone = "clone_id",
  min_q = 0,
  max_q = 4,
  step_q = 1,
  nboot = 100,
  ci = 0.95
)

# Extraer la tabla
df <- hill@diversity

# Agregar columna con índices clásicos
df <- df %>%
  dplyr::mutate(
    indice_clasico = case_when(
      q == 0 ~ d,            # riqueza observada
      q == 1 ~ log(d),       # Shannon clásico H = ln(D1)
      q == 2 ~ 1/d,          # Simpson clásico D = 1/D2
      q == 3 ~ 1/(d^2),      # ∑ p_i^3 = 1/(D3^2)
      q == 4 ~ 1/(d^3)       # ∑ p_i^4 = 1/(D4^3)
    )
  )

print(df)
df_tbl <- as_tibble(df)

# A tibble: 5 x 10
# Groups:   group [1]
  group     q     d  d_sd d_lower d_upper     e e_lower e_upper indice_clasico
  <chr> <dbl> <dbl> <dbl>   <dbl>   <dbl> <dbl>   <dbl>   <dbl>          <dbl>
1 All       0  994.  1.07    992.    996. 1       0.998    1.00        9.94e+2
2 All       1  993.  1.48    990.    996. 1.000   0.997    1.00        6.90e+0
3 All       2  993.  2.13    989.    997. 0.999   0.995    1.00        1.01e-3
4 All       3  992.  3.17    985.    998. 0.998   0.991    1.00        1.02e-6
5 All       4  990.  4.85    980.    999. 0.996   0.986    1.01        1.03e-9


In [145]:
hill_numbers <- function(clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    hill <- alphaDiversity(
        data = clone_counts,
        clone = "clone_id",
        min_q = 0,
        max_q = 4,
        step_q = 1,
        nboot = 100,
        ci = 0.95
    )
    return(hill@diversity)
}
rep_hill_numbers <- hill_numbers(clones)
rep_hill_numbers

group,q,d,d_sd,d_lower,d_upper,e,e_lower,e_upper
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
All,0,993.8500,1.076611,991.7399,995.9601,1.0000000,0.9978768,1.002123
All,1,993.4082,1.489256,990.4893,996.3271,0.9995555,0.9966185,1.002492
All,2,992.7099,2.140001,988.5156,996.9042,0.9988528,0.9946325,1.003073
All,3,991.5831,3.185664,985.3393,997.8269,0.9977191,0.9914366,1.004001
All,4,989.7385,4.884436,980.1652,999.3118,0.9958631,0.9862305,1.005496


In [146]:
richness <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 0) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

richness(rep_hill_numbers)

[1] 993.85

In [147]:
 q1 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q1(rep_hill_numbers)

[1] 993.4082

In [148]:
shannon <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(log(metric_value))
} 

shannon(rep_hill_numbers)

[1] 6.901142

In [149]:
 q2 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q2(rep_hill_numbers)

[1] 992.7099

In [150]:
simpson <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value))
} 

simpson(rep_hill_numbers)

[1] 0.001007344

In [151]:
 q3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 3) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q3(rep_hill_numbers)

[1] 991.5831

In [152]:
d3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value)^2)
} 

d3(rep_hill_numbers)

[1] 1.013315e-06

In [153]:
 q4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q4(rep_hill_numbers)

[1] 989.7385

In [154]:
d4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value)^3)
} 

d4(rep_hill_numbers)

[1] 1.031427e-09

In [155]:
# MÉTRICA CHAO1 y ACE PAQUETE VEGAN
metricas_chao1ace <- clone_counts %>%
  group_by(sample_id) %>% 
  summarise(
    chao1 = estimateR(count)["S.chao1"],
    ace   = estimateR(count)["S.ACE"]
  )

print(metricas_chao1ace)

# A tibble: 1 x 3
  sample_id            chao1    ace
  <chr>                <dbl>  <dbl>
1 repertorio_simulado 82588. 99500.


In [156]:


chao1 <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_chao1 <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      chao1 = as.numeric(vegan::estimateR(count)["S.chao1"]),
      .groups = "drop"
    )%>%
    dplyr::pull(chao1)
  
  return(metricas_chao1[1])
}

# Ejecutar
rep_chao1 <- chao1(clones)
print(rep_chao1)


[1] 82587.5


In [157]:
ace <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_ace <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      ace = as.numeric(vegan::estimateR(count)["S.ACE"]),
      .groups = "drop"
    )%>%
    dplyr::pull(ace)
  
  return(metricas_ace[1])
}

# Ejecutar
rep_ace <- ace(clones)
print(rep_ace)

[1] 99500


In [158]:
# MÉTRICA GINI PAQUETE INEQ
calc_gini <- function(df) {
  ineq::ineq(df$count, type = "Gini")
}
gini_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    gini = calc_gini(cur_data())
  )

print(gini_result)

# A tibble: 1 x 2
  sample_id              gini
  <chr>                 <dbl>
1 repertorio_simulado 0.00497


In [159]:
gini <- function (clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    metrica_gini <- ineq::ineq(clone_counts$count, type = "Gini")
    return(metrica_gini)
}
gini(clones)

[1] 0.004974874

In [160]:
# MÉTRICA PIELOU PAQUETE VEGAN

calc_pielou <- function(df) {
  abund <- df$count
  H <- diversity(abund, index = "shannon")  # Shannon
  S <- specnumber(abund)                    # número de clones
  J <- H / log(S)                           # Pielou
  return(J)
}

pielou_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    pielou = calc_pielou(cur_data())
  )

print(pielou_result)


# A tibble: 1 x 2
  sample_id           pielou
  <chr>                <dbl>
1 repertorio_simulado  1.000


In [161]:
pielou <- function(clones_df){
 
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  H <- vegan::diversity(clone_counts$count, index = "shannon")
  S <- vegan::specnumber(clone_counts$count)
  J <- H / log(S)
  
  return(as.numeric(J))  # 👈 devuelve solo el número
}

pielou(clones)

[1] 0.999722

In [162]:
# MÉTRICA BASHARIN FUNCIONES R+ VEGAN

calc_basharin <- function(df) {
  abund <- df$count
  N <- sum(abund)
  S <- specnumber(abund)
  
  # Casos triviales
  if (N == 0 || S <= 1) return(0)
  
  H <- diversity(abund, index = "shannon")
  print(H)
  basharin <- H + (S - 1) / (2 * N)
  return(basharin)
}

# Aplicar por muestra
basharin_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(basharin = calc_basharin(cur_data()), .groups = "drop")

print(basharin_result)


[1] 6.900824
# A tibble: 1 x 2
  sample_id           basharin
  <chr>                  <dbl>
1 repertorio_simulado     7.40


In [163]:
basharin <- function(clones_df){
  
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  abund <- clone_counts$count
  N <- sum(abund)
  S <- vegan::specnumber(abund)
  
  if (N == 0 || S <= 1) return(0)
  
  # Shannon
  H <- vegan::diversity(abund, index = "shannon")
  
  # Basharin
  basharin_val <- H + (S - 1) / (2 * N)
  
  return(as.numeric(basharin_val))  # 👈 devuelve número puro
}
basharin(clones)

[1] 7.397824

In [164]:
d50_fun <- function(counts) {
  counts <- sort(counts, decreasing = TRUE)
  total <- sum(counts)
  cum <- cumsum(counts)
  which(cum >= 0.5 * total)[1]
}

# Calcular D50
d50_val <- d50_fun(clone_counts$count)
print(d50_val)
d50_result <- tibble::tibble(D50 = d50_val)


[1] 495


In [165]:
d50 <- function(clones_df){
 
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  counts <- sort(clone_counts$count, decreasing = TRUE)
  total  <- sum(counts)
  cum    <- cumsum(counts)
  
  d50_val <- which(cum >= 0.5 * total)[1]
  
  return(as.numeric(d50_val))  # 👈 devuelve número puro
}
d50(clones)

[1] 495

In [166]:
metricas_diversidad <- c(richness= richness(rep_hill_numbers),q1= q1(rep_hill_numbers),shannon= shannon(rep_hill_numbers), q2= q2(rep_hill_numbers), simpson= simpson(rep_hill_numbers), q3= q3(rep_hill_numbers), 
d3= d3(rep_hill_numbers), q4= q4(rep_hill_numbers), d4= d4(rep_hill_numbers), chao1= chao1(clones), ace= ace(clones), gini= gini(clones), pielou= pielou(clones), basharin= basharin(clones), d50= d50(clones))
metricas_diversidad


richness           q1      shannon           q2      simpson           q3 
9.938500e+02 9.934082e+02 6.901142e+00 9.927099e+02 1.007344e-03 9.915831e+02 
          d3           q4           d4        chao1          ace         gini 
1.013315e-06 9.897385e+02 1.031427e-09 8.258750e+04 9.950000e+04 4.974874e-03 
      pielou     basharin          d50 
9.997220e-01 7.397824e+00 4.950000e+02

In [167]:
tabla_diversidad <- bind_rows(metricas_diversidad)
tabla_diversidad$sample_id <- "E_1000seq"
tabla_diversidad

richness,q1,shannon,q2,simpson,q3,d3,q4,d4,chao1,ace,gini,pielou,basharin,d50,sample_id
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
993.85,993.4082,6.901142,992.7099,0.001007344,991.5831,1.013315e-06,989.7385,1.031427e-09,82587.5,99500,0.004974874,0.999722,7.397824,495,E_1000seq


In [168]:
readr::write_tsv(tabla_diversidad, "../results/diversity_metrics/diversity_E_1000seqs.tsv")